# 👶 Genie Ontology 単体の効果を体感する ― 子育てデータ編

**Genie Ontology** は、Genie One / Genie Code が参照する「組織のビジネス文脈の地図」です（2026年11月 GA）。
このノートブックでは **Metric View・テーブルコメント・認証タグ・Genie Agent を一切使わず**、
Ontology の2つの入口だけで Genie One の答えが変わるかを確かめます。

| Step | Ontology の入口 | やること | 特徴 |
|---|---|---|---|
| **A** | **Learned knowledge** | チームが“正しい定義”で書いたSQLを普段どおり実行する（クエリ履歴に残る） | 何もモデリングしなくても、Genie が使われ方から snippet を自動抽出する。**非同期**（反映まで時間がかかることがある） |
| **B** | **Pages**（Modeled knowledge） | 用語の公式定義を Page として公開する | learned より権威が高く、衝突時に優先されて引用される。すぐ効く |

データ本体（テーブル）には **一切手を加えません**。変わるのは Ontology だけです。

参考: [Genie Ontology](https://docs.databricks.com/aws/en/genie/genie-ontology/) / [Pages](https://docs.databricks.com/aws/en/uc-semantics/pages)

## 🎯 シナリオ：子育て記録アプリ「すくすくログ」
約40家庭の育児ログ（睡眠・夜間覚醒・授乳・おむつ）、保育園からの呼び出し、子育て支出。子育て用語は **人によって意味がブレる** のがミソです。

| 用語 | このチームの定義（＝Ontology に覚えてほしいこと） | 素朴な解釈だと… |
|---|---|---|
| **夜泣き** | 22:00〜翌5:00 の夜間覚醒のうち **10分以上** 続いたもの | 短い覚醒まで数える |
| **睡眠充足率** | 1日の総睡眠 ÷ 年齢別推奨睡眠（0歳14h / 1-2歳12h / 3-5歳11h） | 定義できず答えられない |
| **ワンオペ日** | その日の育児記録の **90%以上を同じ1人の大人** が記録した日 | 「ママの記録比率」などで代用 |
| **病児呼び出し** | 保育園の呼び出しのうち **理由=発熱 かつ 37.5℃以上** | けが・嘔吐も含めて全部数える |
| **子育て費用** | `expenses` の **円単位**明細、**大型購入(is_durable)は除外** | 旧集計表 `household_expense_summary`（**千円単位**）を使う |

## 🗺️ 進め方（ウィジェット `step` で切り替え、毎回 *Run all* でOK）
| step | 実行内容 | その後 Genie One で |
|---|---|---|
| `part1` | 素のモックデータ作成 + 正解表 | **⏸ Before**：Q1〜Q5 を質問して記録 |
| `stepA` | 正しい定義のSQLをウェアハウスで繰り返し実行 | **▶ After-A**：しばらく待ってから同じ質問（learned の効果） |
| `stepB` | 用語定義の Page を公開 | **▶ After-B**：同じ質問（Pages の効果） |
| `cleanup` | Page とスキーマを削除 | |

> 💡 正解表は **Python で計算** しています。SQLで計算すると、その「正しい定義入りのSQL」がクエリ履歴に残り、Before の時点で Ontology に学習されてしまう可能性があるためです。

In [ ]:
dbutils.widgets.text("catalog", "workspace", "catalog")
dbutils.widgets.text("schema", "kosodate_log", "schema")
dbutils.widgets.dropdown("step", "part1", ["part1", "stepA", "stepB", "cleanup"], "step")

CAT = dbutils.widgets.get("catalog")
SCH = dbutils.widgets.get("schema")
STEP = dbutils.widgets.get("step")
FQ = f"`{CAT}`.`{SCH}`"
print(f"target = {CAT}.{SCH} / step = {STEP}")

# Part 1 ― モックデータを作る
再現性のため乱数シード固定。期間は **2026-08-01 〜 2026-11-25**。
あえて「罠」を仕込んでいます：
- 夜間覚醒の **回数が最多の子** と **夜泣き(10分以上)が最多の子** は別人
- 呼び出しには「けが」「嘔吐」や 37.5℃未満の発熱も混在
- 支出には耐久財(ベビーカー等)が混在し、さらに **千円単位の旧集計表** `household_expense_summary` が並んでいる

In [ ]:
import random, datetime as dt
from pyspark.sql import types as T

random.seed(20261126)
START, END = dt.date(2026, 8, 1), dt.date(2026, 11, 25)
DAYS = [START + dt.timedelta(d) for d in range((END - START).days + 1)]
AGE_REF = dt.date(2026, 10, 1)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ}")

# ---- 保育施設 ----
nurseries = [
    ("N01", "ひまわり保育園", "認可保育園", "世田谷区"),
    ("N02", "たんぽぽ保育園", "認可保育園", "杉並区"),
    ("N03", "こぐまキッズ", "認可外保育施設", "渋谷区"),
    ("N04", "すみれ幼稚園", "幼稚園", "世田谷区"),
    ("N05", "さくら保育園", "認可保育園", "川崎市中原区"),
    ("N06", "あおぞら幼稚園", "幼稚園", "さいたま市浦和区"),
]
NUR_BY_TYPE = {t: [n[0] for n in nurseries if n[2] == t] for t in {n[2] for n in nurseries}}

# ---- 家庭 ----
PREFS = ["東京都", "東京都", "神奈川県", "埼玉県", "千葉県", "大阪府"]
families = []
for i in range(1, 41):
    ht = random.choices(["共働き", "片働き", "ひとり親"], [0.55, 0.30, 0.15])[0]
    nkids = random.choices([1, 2, 3], [0.5, 0.4, 0.1])[0] if ht != "ひとり親" else random.choice([1, 2])
    families.append((f"F{i:03d}", random.choice(PREFS), ht, nkids))

# ---- 子ども ----
NAMES = ["はると", "ゆい", "そうた", "めい", "りく", "ひなた", "あおい", "こはる", "ゆうと", "さな",
         "いつき", "りん", "そら", "えま", "みなと", "つむぎ", "あさひ", "ほのか", "けんた", "ゆな"]
children = []
cid = 0
for fid, pref, ht, nkids in families:
    ages_days = sorted(random.sample(range(130, 6 * 365 - 30), nkids))
    for a in ages_days:
        cid += 1
        birth = AGE_REF - dt.timedelta(days=a)
        age = a // 365
        if ht in ("共働き", "ひとり親"):
            nur = random.choice(NUR_BY_TYPE["認可保育園"] + NUR_BY_TYPE["認可外保育施設"]) if (age >= 1 or random.random() < 0.6) else None
        else:
            nur = random.choice(NUR_BY_TYPE["幼稚園"]) if age >= 3 else None
        children.append((f"C{cid:03d}", fid, f"{random.choice(NAMES)}{cid:03d}", birth, nur))

def age_years(birth, on=AGE_REF):
    return (on - birth).days // 365

# 罠の子：短い覚醒だらけの子(C_SHORT)と、長い夜泣きの子(C_LONG)
infants = [c for c in children if age_years(c[3]) == 0]
toddlers = [c for c in children if age_years(c[3]) == 1]
C_SHORT = infants[0][0]
C_LONG = (toddlers[0] if toddlers else infants[1])[0]

REC_SLEEP_H = {0: 14, 1: 12, 2: 12, 3: 11, 4: 11, 5: 11}
WAKE_LAMBDA = {0: 2.2, 1: 1.3, 2: 0.8, 3: 0.4, 4: 0.3, 5: 0.2}

fam_by_id = {f[0]: f for f in families}
kids_by_fam = {}
for c in children:
    kids_by_fam.setdefault(c[1], []).append(c)

care_logs = []
lid = 0
def add_log(child_id, logged_by, event_type, ts, dur, ml=None):
    global lid
    lid += 1
    care_logs.append((f"L{lid:07d}", child_id, logged_by, event_type, ts, dur, ml))

def poisson(lam):
    k, p, L = 0, 1.0, pow(2.718281828, -lam)
    while True:
        p *= random.random()
        if p <= L:
            return k
        k += 1

for fid, pref, ht, nkids in families:
    for d in DAYS:
        weekday = d.weekday() < 5
        # その日がワンオペかどうか（家庭タイプで確率が違う）
        if ht == "ひとり親":
            oneope = True
        elif ht == "片働き":
            oneope = random.random() < (0.70 if weekday else 0.30)
        else:
            oneope = random.random() < (0.35 if weekday else 0.10)
        primary = random.choices(["ママ", "パパ"], [0.8, 0.2])[0]
        other = "パパ" if primary == "ママ" else "ママ"
        def who():
            if oneope:
                return primary if random.random() < 0.96 else random.choice([other, "祖父母"]) if ht != "ひとり親" else primary
            return random.choice([primary, other, primary, other, "祖父母"]) if random.random() < 0.9 else primary

        for (child_id, _, _, birth, nur) in kids_by_fam[fid]:
            age = age_years(birth, d)
            if age > 5:
                age = 5
            rec = REC_SLEEP_H[age] * 60
            # 夜の睡眠 + 昼寝
            night = int(rec * random.uniform(0.70, 0.80) * (0.92 if child_id == C_LONG else 1.0))
            nap = int(rec * random.uniform(0.12, 0.25)) if age <= 3 else 0
            add_log(child_id, who(), "睡眠", dt.datetime.combine(d, dt.time(20, random.randint(0, 59))), night)
            if nap:
                add_log(child_id, who(), "睡眠", dt.datetime.combine(d, dt.time(13, random.randint(0, 30))), nap)
            # 夜間覚醒
            lam = WAKE_LAMBDA[age]
            if child_id == C_SHORT:
                lam = 5.0
            if child_id == C_LONG:
                lam = 2.6
            for _ in range(poisson(lam)):
                if child_id == C_SHORT:
                    dur = random.randint(1, 9) if random.random() < 0.93 else random.randint(10, 20)
                elif child_id == C_LONG:
                    dur = random.randint(12, 70)
                else:
                    dur = random.randint(1, 9) if random.random() < 0.55 else random.randint(10, 60)
                hour = random.choice([22, 23, 0, 1, 2, 3, 4])
                ts = dt.datetime.combine(d + dt.timedelta(days=1 if hour < 12 else 0), dt.time(hour, random.randint(0, 59)))
                add_log(child_id, who(), "夜間覚醒", ts, dur)
            # 授乳 / 食事 / おむつ
            if age == 0:
                for h in sorted(random.sample(range(0, 24), random.randint(6, 8))):
                    add_log(child_id, who(), "授乳", dt.datetime.combine(d, dt.time(h, random.randint(0, 59))), random.randint(10, 25), random.choice([80, 100, 120, 140, 160, 200]))
            else:
                for h in (7, 12, 18):
                    add_log(child_id, who(), "食事", dt.datetime.combine(d, dt.time(h, random.randint(0, 40))), random.randint(15, 40))
            if age <= 2:
                for h in sorted(random.sample(range(6, 22), random.randint(4, 7))):
                    add_log(child_id, who(), "おむつ", dt.datetime.combine(d, dt.time(h, random.randint(0, 59))), 3)

# ---- 保育園からの呼び出し ----
nursery_calls = []
ncid = 0
for (child_id, fid, _, birth, nur) in children:
    if not nur:
        continue
    for d in DAYS:
        if d.weekday() >= 5:
            continue
        age = age_years(birth, d)
        p = 0.05 if age <= 1 else 0.025
        if d.month in (10, 11):
            p *= 1.4  # 秋の感染症シーズン
        if random.random() < p:
            ncid += 1
            reason = random.choices(["発熱", "嘔吐", "けが", "体調不良"], [0.55, 0.15, 0.12, 0.18])[0]
            if reason == "発熱":
                temp = round(random.choice([random.uniform(37.0, 37.4), random.uniform(37.5, 39.5), random.uniform(37.5, 39.5)]), 1)
            else:
                temp = round(random.uniform(36.4, 37.3), 1) if random.random() < 0.6 else None
            ht = fam_by_id[fid][2]
            picker = random.choices(["ママ", "パパ", "祖父母", "ファミサポ"], [0.6, 0.25, 0.1, 0.05] if ht != "ひとり親" else [0.8, 0.0, 0.15, 0.05])[0]
            nursery_calls.append((f"K{ncid:05d}", child_id, nur, dt.datetime.combine(d, dt.time(random.randint(9, 16), random.randint(0, 59))), reason, temp, picker))

# ---- 子育て支出（円単位の明細） ----
expenses = []
eid = 0
def add_exp(fid, d, cat, item, amt, durable=False):
    global eid
    eid += 1
    expenses.append((f"E{eid:06d}", fid, d, cat, item, int(amt), durable))

months = sorted({(d.year, d.month) for d in DAYS})
for fid, pref, ht, nkids in families:
    for (y, m) in months:
        d1 = dt.date(y, m, 1)
        for (child_id, _, _, birth, nur) in kids_by_fam[fid]:
            age = age_years(birth, d1)
            if nur:
                ntype = [n for n in nurseries if n[0] == nur][0][2]
                if age < 3:
                    fee = random.randint(25, 60) * 1000 if ntype == "認可保育園" else random.randint(60, 90) * 1000
                    add_exp(fid, d1, "保育料", "月額保育料", fee)
                else:
                    add_exp(fid, d1, "保育料", "給食費・教材費（3歳以上は無償化）", random.randint(4, 9) * 1000)
            if age <= 2:
                add_exp(fid, d1 + dt.timedelta(days=random.randint(0, 20)), "おむつ・ミルク", "おむつ・ミルク", random.randint(6, 15) * 1000)
            add_exp(fid, d1 + dt.timedelta(days=random.randint(0, 25)), "食費", "子どもの食費", random.randint(8, 20) * 1000)
            add_exp(fid, d1 + dt.timedelta(days=random.randint(0, 25)), "衣類", "子ども服", random.randint(2, 9) * 1000)
            if random.random() < 0.4:
                add_exp(fid, d1 + dt.timedelta(days=random.randint(0, 25)), "医療", "小児科・薬", random.randint(0, 3) * 1000 + 500)
            if age >= 3 and random.random() < 0.5:
                add_exp(fid, d1 + dt.timedelta(days=3), "習い事", random.choice(["スイミング", "英語", "体操", "ピアノ"]), random.choice([6000, 8000, 9500]))
        if random.random() < 0.18:
            item, amt = random.choice([("ベビーカー", 58000), ("チャイルドシート", 42000), ("電動自転車", 148000), ("ベビーベッド", 25000), ("学習机", 65000)])
            add_exp(fid, dt.date(y, m, random.randint(1, 25)), "大型購入", item, amt, True)

print(f"families={len(families)} children={len(children)} care_logs={len(care_logs)} calls={len(nursery_calls)} expenses={len(expenses)}")
print(f"罠: 短い覚醒の子={C_SHORT}, 長い夜泣きの子={C_LONG}")

### テーブルとして保存（`step = part1` のときだけ。他の step ではデータに触らない）
コメント・タグ・ビューなどの“意味づけ”は **一切付けません**。

In [ ]:
def save(name, rows, schema):
    spark.createDataFrame(rows, schema).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CAT}.{SCH}.{name}")

if STEP == "part1":
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ}")
    save("nurseries", nurseries, "nursery_id string, nursery_name string, nursery_type string, city string")
    save("families", families, "family_id string, prefecture string, household_type string, num_children int")
    save("children", children, "child_id string, family_id string, nickname string, birth_date date, nursery_id string")
    save("care_logs", care_logs, "log_id string, child_id string, logged_by string, event_type string, start_ts timestamp, duration_min int, amount_ml int")
    save("nursery_calls", nursery_calls, "call_id string, child_id string, nursery_id string, call_ts timestamp, reason string, body_temp double, picked_up_by string")
    save("expenses", expenses, "expense_id string, family_id string, expense_date date, category string, item string, amount_jpy bigint, is_durable boolean")
    # 旧システムの月次集計（千円単位・耐久財込み・11月は未集計）。名前が“それっぽい”のが罠。
    fam_ht = {f[0]: f[2] for f in families}
    legacy = {}
    for (_, fid, d, _, _, amt, _) in expenses:
        if d < dt.date(2026, 11, 1):
            k = (fid, d.strftime("%Y-%m"))
            legacy[k] = legacy.get(k, 0) + amt
    legacy_rows = [(fid, m, fam_ht[fid], round(v / 1000, 1)) for (fid, m), v in sorted(legacy.items())]
    save("household_expense_summary", legacy_rows, "family_id string, month string, household_type string, total_childcare_cost double")
display(spark.sql(f"SHOW TABLES IN {FQ}"))

## 📋 正解表（Python で計算）
`correct` が定義どおりの答え、`naive_answer` が素朴に解釈したときの答えです。Before / After の答え合わせに使います。

In [ ]:
import pandas as pd
from collections import Counter, defaultdict

child = {c[0]: c for c in children}
fam = {f[0]: f for f in families}
is_night = lambda ts: ts.hour >= 22 or ts.hour < 5
in_oct = lambda d: dt.date(2026, 10, 1) <= d < dt.date(2026, 11, 1)

# Q1 夜泣き
all_w, yon = Counter(), Counter()
for (_, cid_, _, ev, ts, dur, _) in care_logs:
    if ev == "夜間覚醒" and in_oct(ts.date()):
        all_w[child[cid_][2]] += 1
        if dur >= 10 and is_night(ts):
            yon[child[cid_][2]] += 1
q1c, q1n = yon.most_common(1)[0], all_w.most_common(1)[0]

# Q2 ワンオペ率
fd = defaultdict(Counter)
for (_, cid_, who_, _, ts, _, _) in care_logs:
    if START <= ts.date() <= END:
        fd[(child[cid_][1], ts.date())][who_] += 1
agg = defaultdict(lambda: [0, 0, 0.0])
for (fid, _), cnt in fd.items():
    tot = sum(cnt.values()); a = agg[fam[fid][2]]
    a[0] += 1; a[1] += max(cnt.values()) / tot >= 0.9; a[2] += cnt["ママ"] / tot
q2c = " / ".join(f"{k} {100*v[1]/v[0]:.1f}%" for k, v in sorted(agg.items()))
q2n = " / ".join(f"{k} ママ記録率{100*v[2]/v[0]:.1f}%" for k, v in sorted(agg.items()))

# Q3 子育て費用（共働き）
fm = {(e[1], e[2].strftime("%Y-%m")) for e in expenses if fam[e[1]][2] == "共働き"}
nd = sum(e[5] for e in expenses if fam[e[1]][2] == "共働き" and not e[6])
wd = sum(e[5] for e in expenses if fam[e[1]][2] == "共働き")
leg = defaultdict(int)
for e in expenses:
    if fam[e[1]][2] == "共働き" and e[2] < dt.date(2026, 11, 1):
        leg[(e[1], e[2].strftime("%Y-%m"))] += e[5]
leg_avg = sum(round(v / 1000, 1) for v in leg.values()) / len(leg)

# Q4 病児呼び出し
oct_calls = [k for k in nursery_calls if in_oct(k[3].date())]
byoji = [k for k in oct_calls if k[4] == "発熱" and k[5] is not None and k[5] >= 37.5]

# Q5 0歳児の睡眠充足率
s_min, s_days = 0, set()
for (_, cid_, _, ev, ts, dur, _) in care_logs:
    if ev == "睡眠" and (ts.date() - child[cid_][3]).days // 365 == 0:
        s_min += dur; s_days.add((cid_, ts.date()))

gt = pd.DataFrame([
    ("Q1 先月(10月)夜泣きが一番多かった子", f"{q1c[0]}（夜泣き{q1c[1]}回）", f"{q1n[0]}（短い覚醒込み{q1n[1]}回）"),
    ("Q2 家庭タイプ別ワンオペ率", q2c, q2n),
    ("Q3 共働き家庭の1家庭あたり月平均子育て費用", f"{nd/len(fm):,.0f} 円", f"耐久財込み {wd/len(fm):,.0f} 円 / 旧表を円と誤読 {leg_avg:,.1f} 円"),
    ("Q4 先月(10月)の病児呼び出し回数", f"{len(byoji)} 回", f"全呼び出し {len(oct_calls)} 回"),
    ("Q5 0歳児の睡眠充足率", f"{100*s_min/(len(s_days)*840):.1f} %（推奨14時間比）", "定義不明で回答不能 / 平均睡眠時間だけ返す"),
], columns=["question", "correct", "naive_answer"])
display(gt)

# ⏸ Before ― Genie One に聞く（`step = part1` の後）
1. 右上のアプリスイッチャーで **Genie One** を開く（または `https://<workspace>/one`）
2. **新しい会話** で次の5問をそのまま聞き、回答・使われたテーブル・SQLを記録（スクショ推奨）

```
Q1. 先月、夜泣きが一番多かった子は誰？
Q2. 家庭タイプ別のワンオペ率を教えて
Q3. 共働き家庭の1家庭あたりの月平均子育て費用は？
Q4. 先月の病児呼び出しは何回あった？
Q5. 0歳児の睡眠充足率は？
```
3. 回答下の **Genie Ontology** をクリックし、この時点では関連する snippet がほぼ無いことを確認

次はウィジェット **`step = stepA`** にして *Run all*。

> Genie One が開けない場合は、ノートブック右上の **Genie Code** に同じ質問をしても同じ Ontology が使われます。

In [ ]:
if STEP == "part1":
    dbutils.notebook.exit("Part 1 完了 → Genie One で Before の質問をしてください")

# Step A ― Learned knowledge：正しいSQLを“普段使い”する
Ontology は **クエリ履歴に残ったSQL**（dbt など外部ツール経由も含む）から snippet を自動抽出します。
ここでは「チームのアナリストが定義どおりに書いた日常クエリ」を **SQLウェアハウス** で数回ずつ実行し、クエリ履歴に残します。

- テーブルには何も足しません（コメントもビューも作らない）
- 定義は **SQLそのもの** と、先頭の **SQLコメント** にだけ書かれています
- 抽出のタイミングは公開されていません。数十分〜数時間おいてから After-A を試してください

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState
w = WorkspaceClient()

T = lambda t: f"{CAT}.{SCH}.{t}"
team_queries = {
"夜泣きランキング（先月）": f"""
-- 夜泣き回数ランキング（先月）
-- 定義: 夜泣き = 22:00〜翌5:00 の「夜間覚醒」のうち 10分以上続いたもの。10分未満の短い覚醒は夜泣きに数えない。
SELECT c.nickname AS child, COUNT(*) AS yonaki_count
FROM {T('care_logs')} l JOIN {T('children')} c ON l.child_id = c.child_id
WHERE l.event_type = '夜間覚醒'
  AND l.duration_min >= 10
  AND (hour(l.start_ts) >= 22 OR hour(l.start_ts) < 5)
  AND l.start_ts >= add_months(date_trunc('MONTH', current_date()), -1)
  AND l.start_ts <  date_trunc('MONTH', current_date())
GROUP BY c.nickname ORDER BY yonaki_count DESC""",
"夜泣き回数の月次推移": f"""
-- 夜泣き回数（月次）。夜泣き = 22-5時の夜間覚醒で duration_min >= 10 のみ
SELECT date_trunc('MONTH', l.start_ts) AS month, COUNT(*) AS yonaki_count
FROM {T('care_logs')} l
WHERE l.event_type = '夜間覚醒' AND l.duration_min >= 10 AND (hour(l.start_ts) >= 22 OR hour(l.start_ts) < 5)
GROUP BY 1 ORDER BY 1""",
"家庭タイプ別ワンオペ率": f"""
-- ワンオペ率（家庭タイプ別）
-- 定義: ワンオペ日 = その日の育児記録(care_logs)の90%以上を同じ1人の大人(logged_by)が記録した日。ワンオペ率 = ワンオペ日 / 記録のある日。
WITH per_person AS (
  SELECT c.family_id, to_date(l.start_ts) AS log_date, l.logged_by, COUNT(*) AS n
  FROM {T('care_logs')} l JOIN {T('children')} c ON l.child_id = c.child_id
  GROUP BY 1, 2, 3
), per_day AS (
  SELECT family_id, log_date, MAX(n) / SUM(n) AS top_share FROM per_person GROUP BY 1, 2
)
SELECT f.household_type, ROUND(100 * AVG(CASE WHEN d.top_share >= 0.9 THEN 1 ELSE 0 END), 1) AS oneope_rate_pct
FROM per_day d JOIN {T('families')} f ON d.family_id = f.family_id
GROUP BY f.household_type ORDER BY oneope_rate_pct DESC""",
"月平均子育て費用": f"""
-- 1家庭あたり月平均子育て費用（円）
-- 定義: expenses（円単位の明細）から算出し、大型購入(is_durable = true: ベビーカー等)は除外する。
-- household_expense_summary は旧システムの千円単位集計なので使わない。
SELECT f.household_type,
       ROUND(SUM(CASE WHEN NOT e.is_durable THEN e.amount_jpy ELSE 0 END)
             / COUNT(DISTINCT e.family_id, date_format(e.expense_date, 'yyyy-MM'))) AS avg_monthly_childcare_cost_jpy
FROM {T('expenses')} e JOIN {T('families')} f ON e.family_id = f.family_id
GROUP BY f.household_type""",
"病児呼び出し（月次）": f"""
-- 病児呼び出し回数（月次）
-- 定義: 病児呼び出し = nursery_calls のうち reason = '発熱' かつ body_temp >= 37.5（園の呼び出し基準）。けが・嘔吐・体調不良・37.5℃未満は含めない。
SELECT date_trunc('MONTH', k.call_ts) AS month,
       COUNT_IF(k.reason = '発熱' AND k.body_temp >= 37.5) AS sick_call_count
FROM {T('nursery_calls')} k
GROUP BY 1 ORDER BY 1""",
"年齢別睡眠充足率": f"""
-- 睡眠充足率（年齢別）
-- 定義: 睡眠充足率 = 1日の総睡眠(event_type='睡眠' の duration_min 合計) ÷ 年齢別推奨睡眠（0歳 14h=840分, 1-2歳 12h=720分, 3-5歳 11h=660分）
WITH d AS (
  SELECT l.child_id, to_date(l.start_ts) AS log_date,
         floor(datediff(to_date(l.start_ts), c.birth_date) / 365) AS age_years,
         SUM(l.duration_min) AS sleep_min
  FROM {T('care_logs')} l JOIN {T('children')} c ON l.child_id = c.child_id
  WHERE l.event_type = '睡眠'
  GROUP BY 1, 2, 3
)
SELECT age_years,
       ROUND(100 * SUM(sleep_min) / SUM(CASE WHEN age_years = 0 THEN 840 WHEN age_years <= 2 THEN 720 ELSE 660 END), 1) AS sleep_sufficiency_pct
FROM d GROUP BY age_years ORDER BY age_years""",
}

if STEP == "stepA":
    wh = next(x for x in w.warehouses.list())
    print("warehouse:", wh.name)
    REPEAT = 4  # “よく使われるクエリ”にするため複数回
    for i in range(REPEAT):
        for title, sql in team_queries.items():
            r = w.statement_execution.execute_statement(warehouse_id=wh.id, statement=sql, wait_timeout="50s")
            while r.status.state in (StatementState.PENDING, StatementState.RUNNING):
                import time; time.sleep(2)
                r = w.statement_execution.get_statement(r.statement_id)
            assert r.status.state == StatementState.SUCCEEDED, (title, r.status)
            if i == REPEAT - 1:
                print(f"■ {title}: {r.result.data_array[:6]}")

# ▶ After-A ― learned knowledge の効果を見る
1. しばらく待つ（目安：数十分〜数時間。抽出タイミングは非公開）
2. Genie One で **新しい会話** を開き Q1〜Q5 を再質問
3. 回答下の **Genie Ontology** → snippet を開き、**出所が上のクエリ（query history）** になっているかを確認
4. 正解表と比較。ここで変わった分が **learned knowledge 単体** の効果です

次は **`step = stepB`** で *Run all*。

In [ ]:
if STEP == "stepA":
    dbutils.notebook.exit("Step A 完了 → 少し待ってから Genie One で After-A を確認してください")

# Step B ― Pages：用語の公式定義を書く（Modeled knowledge）
Page は「ビジネス概念の公式定義」で、Genie One は Page を **learned knowledge より優先** し、回答で引用します。
Free Edition にも既定ドメイン **General** があるので、そこに Pages API（`POST /api/2.0/knowledge/pages`）で公開します。
UIで作るなら **Discover → Pages → Create page** に下の内容をコピペしても同じです。

In [ ]:
pages = [
  ("夜泣き", ["夜泣き回数", "よなき", "yonaki", "night crying"],
   "子どもが夜中に起きて泣く・ぐずること（すくすくログ上の定義）",
   f"""## 定義
**夜泣き** = `{T('care_logs')}` で `event_type = '夜間覚醒'` のうち、**22:00〜翌5:00** に始まり **10分以上**（`duration_min >= 10`）続いたもの。

- 10分未満の短い覚醒（寝返り・寝言レベル）は夜泣きに**含めない**
- 「夜泣きが多い子」は夜泣き回数で比較する。夜間覚醒の総数ではない
"""),
  ("ワンオペ日 / ワンオペ率", ["ワンオペ", "ワンオペ育児", "ワンオペ率", "一人育児"],
   "1人の大人だけで育児を回した日",
   f"""## 定義
**ワンオペ日** = 家庭(family_id)×日で、その日の `{T('care_logs')}` の記録の **90%以上を同じ1人の大人**（`logged_by`）が記録した日。

**ワンオペ率** = ワンオペ日数 ÷ 記録のある日数。家庭タイプ別に見るときは `{T('families')}.household_type` で集計する。

- 「ママが記録した割合」はワンオペ率**ではない**
"""),
  ("病児呼び出し", ["お迎えコール", "発熱呼び出し", "保育園からの呼び出し", "sick call"],
   "保育園から発熱でお迎えを求められた呼び出し",
   f"""## 定義
**病児呼び出し** = `{T('nursery_calls')}` のうち **`reason = '発熱'` かつ `body_temp >= 37.5`**（園の呼び出し基準）。

- けが・嘔吐・体調不良、37.5℃未満の発熱は**含めない**
"""),
  ("子育て費用", ["育児費用", "子育てコスト", "月平均子育て費用", "childcare cost"],
   "子どもにかかった月々の費用（円）",
   f"""## 定義
**子育て費用** は `{T('expenses')}`（**円単位**の明細）の `amount_jpy` 合計。**大型購入（`is_durable = true`、ベビーカー・電動自転車など）は除外**する。

**1家庭あたり月平均** = 子育て費用 ÷ (家庭×月 の数)。

- ⚠️ `{T('household_expense_summary')}` は旧システムの集計で **千円単位**・耐久財込み・11月以降なし。子育て費用の計算に**使わない**
"""),
  ("睡眠充足率", ["睡眠充足度", "睡眠足りてる", "sleep sufficiency"],
   "年齢別の推奨睡眠時間に対して実際にどれだけ眠れたか",
   f"""## 定義
**睡眠充足率** = 1日の総睡眠（`{T('care_logs')}` の `event_type = '睡眠'` の `duration_min` 合計、昼寝含む） ÷ 年齢別推奨睡眠。

| 年齢（`birth_date` からの満年齢） | 推奨睡眠 |
|---|---|
| 0歳 | 14時間（840分） |
| 1〜2歳 | 12時間（720分） |
| 3〜5歳 | 11時間（660分） |
"""),
]

def general_domain():
    for d in w.api_client.do("GET", "/api/2.0/domains").get("domains", []):
        if d["tag_key"] == "system.general_domain":
            return d["name"]

def list_my_pages(domain):
    out, tok = [], None
    while True:
        q = {"page_size": 100, "filter": f'domain = "{domain}"'}
        if tok: q["page_token"] = tok
        r = w.api_client.do("GET", "/api/2.0/knowledge/pages", query=q)
        out += r.get("pages", [])
        tok = r.get("next_page_token")
        if not tok: return out

PAGE_TITLES = {p[0] for p in pages}

if STEP == "stepB":
    dom = general_domain()
    existing = {p["display_name"]: p["name"] for p in list_my_pages(dom)}
    for title, syn, desc, body in pages:
        payload = {"domain": dom, "display_name": title, "synonyms": syn, "description": desc, "body": body, "state": "PUBLISHED"}
        if title in existing:
            w.api_client.do("PATCH", f"/api/2.0/knowledge/{existing[title]}", query={"update_mask": "body,synonyms,description,state"}, body=payload)
            print("updated  ", title)
        else:
            r = w.api_client.do("POST", "/api/2.0/knowledge/pages", body=payload)
            print("published", title, r["name"])

# ▶ After-B ― Pages の効果を見る
1. Genie One で **新しい会話** を開き Q1〜Q5 を再質問（Pages は比較的すぐ効きます）
2. 回答のシンキングステップ／引用に **Page「夜泣き」などが出てくるか** を確認
3. 定義そのものも聞いてみる：`夜泣きってどう定義してる？` `子育て費用に household_expense_summary を使っていい？`
4. Before → After-A → After-B で正解表にどれだけ近づいたかを比較

### 🧪 発展：衝突解決を見る
Page「夜泣き」の本文を「**5分以上**」に書き換えて `step = stepB` で再実行 → Step A の learned（10分以上）と衝突したとき、Genie が **Page（modeled）を優先** するかを見る。

片付けは **`step = cleanup`** で *Run all*。

In [ ]:
if STEP == "stepB":
    dbutils.notebook.exit("Step B 完了 → Genie One で After-B を確認してください")

In [ ]:
if STEP == "cleanup":
    dom = general_domain()
    for p in list_my_pages(dom):
        if p["display_name"] in PAGE_TITLES:
            w.api_client.do("DELETE", f"/api/2.0/knowledge/{p['name']}")
            print("deleted page", p["display_name"])
    spark.sql(f"DROP SCHEMA IF EXISTS {FQ} CASCADE")
    print("dropped", FQ)